# Green Gentrification Index — Data Exploration

**Objective:** Load all 16 Green Sentinel stations through the unified data pipeline, inspect basic statistics, and visualize anomalies and gaps.

Data: `monitoring_2026-05-21_2026-06-19` (30 days, hourly). Also loads DKV bus-stop data.

In [ ]:
import logging, sys, warnings
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.pipeline import DataPipeline
from src.loaders import GreenSentinelLoader, DKVLoader

In [ ]:
pipeline = DataPipeline()
pipeline.add_dataset(GreenSentinelLoader()).add_dataset(DKVLoader())
pipeline.load_all()

gs = pipeline.get_dataset('green_sentinel')
dkv = pipeline.get_dataset('dkv')

print('Green Sentinel:', gs.shape)
print('DKV bus stops:', dkv.shape)

## Basic statistics

In [ ]:
print(gs.dtypes.to_string())
print()
print('Missing values per column:')
print(gs.isna().sum())

In [ ]:
print('Stations:')
print(sorted(gs['station'].unique()))
print()
print('Measurement types:')
print(sorted(gs['measurement_type'].unique()))
print()
print('Date range:', gs['timestamp'].min(), '->', gs['timestamp'].max())
print('Readings per station:')
print(gs.groupby('station').size())

## Anomalies and gaps

Check for negative values (sensor drift), `NaN` gaps, and count readings per measurement type per day.

In [ ]:
neg = gs[gs['value'] < 0]
print(f'Negative anomalies: {len(neg)} ({len(neg)/len(gs)*100:.3f}%)')
if len(neg):
    print(neg.groupby('measurement_type').size().to_string())

nulls = gs['value'].isna().sum()
print(f'\nMissing values: {nulls} ({nulls/len(gs)*100:.3f}%)')
print('Missing by measurement type:')
print(gs[gs['value'].isna()].groupby('measurement_type').size().to_string())

In [ ]:
pm25 = gs[gs['measurement_type'] == 'PM2.5']
pm25_wide = pm25.pivot_table(index='timestamp', columns='station', values='value')

fig, ax = plt.subplots(figsize=(14, 4))
pm25_wide.plot(ax=ax, legend=False, alpha=0.7)
ax.set_title('PM2.5 daily readings per station')
ax.set_ylabel('µg/m³')
plt.tight_layout(); plt.show()

In [ ]:
corr = pm25_wide.corr()
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Station-to-station correlation of PM2.5')
plt.tight_layout(); plt.show()

## Data quality report

The pipeline records a per-loader quality report with anomaly statistics.

In [ ]:
import json
print(json.dumps(pipeline.get_metadata()['green_sentinel']['quality_report'], indent=2, default=str))